In [1]:
import os
from dotenv import load_dotenv
import chromadb


In [10]:
chroma_client = chromadb.HttpClient(host='localhost', port="8000")

In [58]:
colls = chroma_client.list_collections()
colls

[Collection(name=openai_collection), Collection(name=my_collection)]

In [ ]:

chroma_client.delete_collection(name="my_collection")
collection = chroma_client.create_collection(name="my_collection")

collection.add(
    embeddings=[[1.2, 2.3, 4.5], [6.7, 8.2, 9.2]],
    documents=["This is a document", "This is another document"],
    metadatas=[{"source": "my_source"}, {"source": "my_source"}],
    ids=["id1", "id2"]
)


In [ ]:
results = collection.query(
    query_texts=["This is a query document"],
    n_results=2
)

In [14]:

# Load
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()


In [16]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)


In [39]:
len(all_splits)

all_splits[4]

docs = [item.page_content for item in all_splits]
metas = [item.metadata for item in all_splits]

In [45]:

# Add to vectorDB
# vectorstore = Chroma.from_documents(documents=all_splits,
#                                     collection_name="rag-chroma",
#                                     embedding=OpenAIEmbeddings(),
#                                     )
# retriever = vectorstore.as_retriever()


# Example setup of the client to connect to your chroma server
# client = chromadb.HttpClient(host="localhost", port="8000")

chroma_vectorstore = Chroma(
    collection_name="my_collection", embedding_function=OpenAIEmbeddings(), client=chroma_client
)

retriever = chroma_vectorstore.as_retriever()

from chromadb.utils import embedding_functions
from dotenv import load_dotenv
import os

load_dotenv()
openai_ef = embedding_functions.OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"])

collection = chroma_client.get_or_create_collection(name="openai_collection", embedding_function=openai_ef)
collection.count()
ids = list(range(len(all_splits)))

s_ids = [str(i) for i in ids]


In [47]:


collection.add(ids=s_ids, documents=docs, metadatas=metas) # type: ignore


In [54]:
collection.count()

# collection.get(ids=["1","2","3"], include=["metadatas", "documents"])

results = collection.query(query_texts="What is an agent", n_results=4)

In [55]:
results

{'ids': [['2', '75', '72', '6']],
 'distances': [[0.41335517168045044,
   0.4185360074043274,
   0.4234216809272766,
   0.4324064254760742]],
 'embeddings': None,
 'metadatas': [[{'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:',
    'language': 'en',
    'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'title': "LLM Powered Autonomous Agents | Lil'Log"},
   {'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, G